# Pokémon Stats: K-means Clustering

Explore whether Pokémon with similar battle stats form natural groups. This notebook loads `Pokemon.csv`, scales the six base stats, compares candidate cluster counts, fits K-means, and visualizes the results.

**Colab:** Upload `Pokemon.csv` when prompted in the first code cell. The CSV is not embedded in this notebook, so you can reuse the notebook with another copy of the dataset.

## 1. Load the data

Run this cell and choose `Pokemon.csv` from your computer. If the CSV is already in the Colab session, the upload prompt is skipped.

In [ ]:
import os
import pandas as pd

CSV_PATH = "Pokemon.csv"
if not os.path.exists(CSV_PATH):
    from google.colab import files
    uploaded = files.upload()
    CSV_PATH = next(iter(uploaded))

pokemon = pd.read_csv(CSV_PATH)
print(f"Rows: {len(pokemon):,} | Columns: {pokemon.shape[1]}")
display(pokemon.head())
print("\nColumns:", pokemon.columns.tolist())

## 2. Choose and scale the features

We’ll cluster on **HP, Attack, Defense, Sp. Atk, Sp. Def, and Speed**. Names and types are labels, while `Total` is the sum of these stats, so including it would count the same information twice. Scaling puts every stat on a comparable footing.

K-means uses Euclidean distance and is sensitive to scale. Scaling is therefore an important part of this analysis.

In [ ]:
from sklearn.preprocessing import StandardScaler

stat_columns = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
missing_columns = [col for col in stat_columns if col not in pokemon.columns]
if missing_columns:
    raise ValueError(f"Expected stat columns are missing: {missing_columns}")

# Keep the six stats as numeric values and exclude rows with incomplete stats.
X_raw = pokemon[stat_columns].apply(pd.to_numeric, errors="coerce")
valid_rows = X_raw.notna().all(axis=1)
X_raw = X_raw.loc[valid_rows].copy()
pokemon_clean = pokemon.loc[valid_rows].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
print(f"Using {len(X_raw)} Pokémon and {len(stat_columns)} standardized stats.")

## 3. How many clusters?

The **elbow curve** shows within-cluster sum of squared distances (inertia). Lower is better, but inertia always decreases as more clusters are added. Look for a bend where improvements start to taper off. The **silhouette score** compares how close each point is to its own cluster versus neighboring clusters; higher is better, with values between −1 and 1. Neither metric determines a single objectively correct answer.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

max_k = min(10, len(X_scaled) - 1)
k_values = range(2, max_k + 1)
inertias, silhouettes = [], []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(k_values), inertias, marker="o", color="#731DD8")
axes[0].set(title="Elbow method", xlabel="Number of clusters (k)", ylabel="Inertia")
axes[0].grid(alpha=.25)
axes[1].plot(list(k_values), silhouettes, marker="o", color="#48A9A6")
axes[1].set(title="Silhouette score", xlabel="Number of clusters (k)", ylabel="Score")
axes[1].grid(alpha=.25)
plt.tight_layout()
plt.show()

## 4. Fit K-means

Choose `K` after reviewing the plots. Start with 4, then try another value and compare whether the cluster profiles are easier to interpret. Each run uses a fixed random seed for reproducibility.

In [ ]:
K = 4  # Change this value to explore a different number of groups

if not 2 <= K <= max_k:
    raise ValueError(f"Choose K between 2 and {max_k}.")

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
pokemon_clean["Cluster"] = kmeans.fit_predict(X_scaled)
print("Pokémon in each cluster:")
display(pokemon_clean["Cluster"].value_counts().sort_index().rename("Count").to_frame())

## 5. Understand the clusters

The table below shows average **original stat values**, not standardized values. Use it to describe each group (for example, a high-Speed group or a high-Defense group). Cluster numbers are arbitrary labels; cluster 0 is not inherently better than cluster 1.

In [ ]:
cluster_profile = pokemon_clean.groupby("Cluster")[stat_columns].mean().round(1)
cluster_profile["Pokémon count"] = pokemon_clean.groupby("Cluster").size()
display(cluster_profile)

# Inspect sample Pokémon in each cluster.
name_column = "Name" if "Name" in pokemon_clean.columns else None
for cluster_id in sorted(pokemon_clean["Cluster"].unique()):
    group = pokemon_clean[pokemon_clean["Cluster"] == cluster_id]
    names = group[name_column].head(10).tolist() if name_column else group.index[:10].tolist()
    print(f"Cluster {cluster_id} examples ({len(group)} total): {', '.join(map(str, names))}")

## 6. Visualize the clusters

The left chart uses **PCA** to project the six-dimensional stats into two dimensions. It preserves as much overall variation as possible, but some separation can be lost in this projection. The right chart plots Attack against Defense directly. In both charts, color indicates the assigned cluster.

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)
plot_data = pd.DataFrame({
    "PC1": coords[:, 0], "PC2": coords[:, 1],
    "Cluster": pokemon_clean["Cluster"].to_numpy(),
    "Attack": X_raw["Attack"].to_numpy(),
    "Defense": X_raw["Defense"].to_numpy()
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter = axes[0].scatter(plot_data["PC1"], plot_data["PC2"],
                          c=plot_data["Cluster"], cmap="tab10", alpha=.75, s=28)
axes[0].set(title=f"PCA view (explained variance: {pca.explained_variance_ratio_.sum():.1%})",
            xlabel="Principal component 1", ylabel="Principal component 2")
axes[0].grid(alpha=.2)
axes[1].scatter(plot_data["Attack"], plot_data["Defense"],
                c=plot_data["Cluster"], cmap="tab10", alpha=.75, s=28)
axes[1].set(title="Attack vs Defense", xlabel="Attack", ylabel="Defense")
axes[1].grid(alpha=.2)
handles, labels = scatter.legend_elements()
axes[0].legend(handles, [f"Cluster {i}" for i in range(K)], title="Groups", loc="best")
axes[1].legend(handles, [f"Cluster {i}" for i in range(K)], title="Groups", loc="best")
plt.tight_layout()
plt.show()

## 7. Optional: compare clusters with the Legendary label

`Legendary` was **not** used to create the clusters. This table is an after-the-fact check: do some clusters contain a larger share of legendary Pokémon? A pattern here does not mean K-means learned the label; it only describes how that label is distributed across the stat-based groups.

In [ ]:
if "Legendary" in pokemon_clean.columns:
    comparison = pd.crosstab(pokemon_clean["Cluster"], pokemon_clean["Legendary"], normalize="index")
    comparison = comparison.rename(columns={False: "Not legendary share", True: "Legendary share"})
    display(comparison.round(3))
    comparison.plot(kind="bar", stacked=True, figsize=(8, 4),
                    color=["#48A9A6", "#731DD8"], alpha=.9)
    plt.title("Legendary share within each stat-based cluster")
    plt.xlabel("Cluster")
    plt.ylabel("Share")
    plt.legend(title="Label")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("This CSV has no Legendary column, so the optional comparison was skipped.")

## 8. Reflection

- Which value of `K` seems easiest to explain from the elbow and silhouette plots?
- What battle-stat pattern defines each cluster?
- Does the PCA picture agree with the cluster profile table?
- How do results change if you use only Attack and Defense, or include `Total`?
- What information is lost when six stats are projected onto two PCA axes?

**Reminder:** K-means groups points by distance. It does not know Pokémon types, evolution, roles, or which Pokémon are “strongest.” Results depend on feature choices, scaling, and `K`.